In [1]:
import folium
import numpy as np
import pandas as pd
import webbrowser
from math import sin, cos, sqrt, atan2, radians


In [2]:
edges = (
    pd.read_csv("../data/edges.csv")
    .rename(columns=lambda x: x.strip())[["# source", "target", "distance"]]
    .drop_duplicates()
)
edges = edges.set_index(["# source", "target"])
nodes = pd.read_csv("../data/nodes.csv").rename(columns=lambda x: x.strip())[
    ["name", "city", "country", "latitude", "longitude", "altitude"]
]
nodes = pd.merge(nodes, pd.read_csv("../data/continents.csv"), on="country")

# solwesi ma zle suradnice a zle distance tympadom: -12.170638697187435,26.36774594217987

In [3]:
class RouteMap:
    def __init__(self, center=[20, 0], zoom_start=2, tiles="cartodb positron"):
        self.m = folium.Map(
            location=center, zoom_start=zoom_start, tiles=tiles, prefer_canvas=True
        )
        self.route_color = "#546E7A"  # Jednotná šedomodrá farba

    def add_nodes(self, df, show_text=False):
        continents_list = [
            "Africa",
            "Asia",
            "Europe",
            "North America",
            "Oceania",
            "South America",
        ]

        for idx, r in df.iterrows():
            # Červený bod letiska
            folium.CircleMarker(
                [r["latitude"], r["longitude"]],
                radius=6,
                color="white",
                weight=2,
                fill=True,
                fill_color="#DD0A0A",
                fill_opacity=1.0,
                zorder=100,
            ).add_to(self.m)

            if show_text:
                raw_name = r.get("city", r.get("name", str(idx)))
                name = (
                    raw_name.replace("International Airport", "")
                    .replace("Airport", "")
                    .strip()
                    .rstrip(",")
                )
                country = r.get("country", "")

                # Rozlíšenie pozadia
                is_continent = raw_name in continents_list
                bg_color = "#115E07" if is_continent else "white"
                text_color = "white" if is_continent else "black"
                badge_txt = "CONTINENT" if is_continent else country

                # Zoberieme anchor priamo z riadku (predtým naplnený cez index-mapping)
                anchor = r.get("anchor", (79, 40))

                html_content = f"""
                    <div style="background: {bg_color}; color: {text_color}; border: 2px solid {self.route_color}; 
                                border-radius: 8px; width: 180px; box-shadow: 0 4px 8px rgba(0,0,0,0.2); 
                                overflow: hidden; display: flex; flex-direction: column; font-family: Arial;">
                        <div style="padding: 6px; text-align: center; font-size: 11pt; font-weight: 900; text-transform: uppercase;">
                            {name}
                        </div>
                        <div style="background: {self.route_color}; color: white; font-size: 8pt; font-weight: bold; 
                                    padding: 2px 0; text-align: center; text-transform: uppercase;">
                            {badge_txt}
                        </div>
                    </div>
                """

                folium.Marker(
                    [r["latitude"], r["longitude"]],
                    icon=folium.DivIcon(
                        icon_size=(180, 80), icon_anchor=anchor, html=html_content
                    ),
                ).add_to(self.m)

    def add_route(self, df, value_show=True):
        for _, row in df.iterrows():
            s_lat, s_lon = map(float, str(row["start"]).split(","))
            e_lat, e_lon = map(float, str(row["end"]).split(","))

            # 1. Výpočet virtuálneho konca pre matematiku (uhol, stred)
            v_e_lon = e_lon
            needs_split = False
            
            if e_lon - s_lon > 180:
                v_e_lon -= 360
                needs_split = True
            elif e_lon - s_lon < -180:
                v_e_lon += 360
                needs_split = True

            # Smer šípky a vzdialenosť
            angle = np.degrees(np.arctan2(e_lat - s_lat, v_e_lon - s_lon))
            dist = int(row["distance"])

            if angle > 90 or angle < -90:
                display_angle = angle - 180
                label_html = f"← {dist} km"
            else:
                display_angle = angle
                label_html = f"{dist} km →"

            # 2. Vykreslenie čiary (so splitom pre antimeridián)
            if needs_split:
                # Vypočítame, kde presne čiara pretne okraj mapy (180/-180)
                edge_lon = -180.0 if v_e_lon < -180 else 180.0
                
                # Lineárna interpolácia zemepisnej šírky na hrane mapy
                ratio = (edge_lon - s_lon) / (v_e_lon - s_lon)
                intersect_lat = s_lat + (e_lat - s_lat) * ratio

                # Segment 1: Od štartu k hrane
                folium.PolyLine(
                    [[s_lat, s_lon], [intersect_lat, edge_lon]],
                    color=self.route_color, weight=4, opacity=0.6
                ).add_to(self.m)

                # Segment 2: Od opačnej hrany k cieľu
                folium.PolyLine(
                    [[intersect_lat, -edge_lon], [e_lat, e_lon]],
                    color=self.route_color, weight=4, opacity=0.6
                ).add_to(self.m)
            else:
                # Klasická čiara bez skoku cez antimeridián
                folium.PolyLine(
                    [[s_lat, s_lon], [e_lat, e_lon]],
                    color=self.route_color, weight=4, opacity=0.6
                ).add_to(self.m)

            # 3. Umiestnenie popisku
            if value_show:
                # Stred na virtuálnej čiare
                m_lat = (s_lat + e_lat) / 2
                m_lon = (s_lon + v_e_lon) / 2
                
                # Normalizácia m_lon späť do rozsahu -180 až 180 pre Marker
                final_m_lon = (m_lon + 180) % 360 - 180
                
                folium.Marker(
                    [m_lat, final_m_lon],
                    icon=folium.DivIcon(html=f"""
                        <div style="transform: rotate({-display_angle}deg); color: {self.route_color}; 
                                    font-weight: bold; font-size: 13pt; white-space: nowrap; 
                                    text-shadow: 1px 1px white; background: rgba(255,255,255,0.4); 
                                    padding: 2px 5px; border-radius: 4px;">
                            {label_html}
                        </div>"""),
                ).add_to(self.m)

    def showMap(self, filename="mapa_trasy.html"):
        self.m.save(filename)
        webbrowser.open(filename)

In [4]:
def make_start_end(edges_df):
    new_edges = edges_df.copy().reset_index()
    new_edges[["start", "end"]] = (
        new_edges[["# source", "target"]]
        .apply(
            lambda row: f"{nodes.loc[row['# source'],'latitude']},"
            + f"{nodes.loc[row['# source'],'longitude']};"
            + f"{nodes.loc[row['target'],'latitude']},"
            + f"{nodes.loc[row['target'],'longitude']}",
            axis=1,
        )
        .str.split(";", expand=True)
    )
    return new_edges

In [5]:
# 1. Zoznam indexov trasy
node_indices = [
    2317,
    2287,
    2330,
    2292,
    57,
    86,
    125,
    770,
    461,
    464,
    468,
    3205,
    466,
    465,
]

# 2. Manuálny mapping pre tieto konkrétne letiská (z tvojho predošlého zadania)
route_mapping = {
    2317: (79, 75),  # Peawanuck
    2287: (220, 42),  # Attawapiskat
    2330: (-20, 47),  # Kashechewan
    2292: (200, -2),  # Fort Albany
    57: (-20, 7),  # Moosonee
    86: (180, -15),  # Timmins
    125: (79, -10),  # Toronto (Pearson)
    770: (79, 75),  # Istanbul (Atatürk)
    461: (180, 40),  # Kinshasa (Ndjili)
    464: (200, 45),  # Kisangani
    468: (130, -5),  # Goma
    3205: (-10, 10),  # Beni
    466: (-5, 50),  # Bunia
    465: (90, 90),  # Isiro (Matari)
}

# 3. Vytvorenie DataFrame pre uzly (Nodes)
route_nodes = nodes.loc[node_indices].copy()
# 1. Priradenie kotvy cez index (idx je index v DataFrame, ktorý zodpovedá route_mapping)
route_nodes["anchor"] = route_nodes.index.map(route_mapping)

# 2. Ak by nejaký index v slovníku chýbal, dáme mu stredovú kotvu
route_nodes["anchor"] = route_nodes["anchor"].fillna(
    {i: (79, 40) for i in route_nodes.index}
)

# 3. Vykreslenie
rm = RouteMap()
rm.add_nodes(route_nodes, show_text=True)


route_edges = pd.DataFrame(
    zip(node_indices, node_indices[1:]), columns=["# source", "target"]
)
route_edges = edges.loc[zip(node_indices, node_indices[1:])]

route_edges = make_start_end(route_edges)
rm.add_route(route_edges, False)
rm.showMap("./vizs/route_map.html")

In [6]:
route_edges

,# source,target,distance,start,end
0,2317,2287,302.225602,"54.98809814453125,-85.44329833984375","52.9275016784668,-82.43190002441406"
1,2287,2330,87.987282,"52.9275016784668,-82.43190002441406","52.282501220703125,-81.67780303955078"
2,2330,2292,9.114073,"52.282501220703125,-81.67780303955078","52.20140075683594,-81.6968994140625"
3,2292,57,126.003148,"52.20140075683594,-81.6968994140625","51.29109954833984,-80.60780334472656"
4,57,86,307.661795,"51.29109954833984,-80.60780334472656","48.5696983337,-81.376701355"
5,86,125,560.551645,"48.5696983337,-81.376701355","43.6772003174,-79.63059997559999"
6,125,770,8201.411309,"43.6772003174,-79.63059997559999","40.976898,28.8146"
7,770,461,5225.914053,"40.976898,28.8146","-4.38575,15.4446"
8,461,464,1225.538438,"-4.38575,15.4446","0.481638997793,25.3379993439"
9,464,468,495.485122,"0.481638997793,25.3379993439","-1.670809984207153,29.238500595092773"


In [7]:
# 1. Zoznam indexov trasy
node_indices = [2909, 863, 860, 868, 1744, 1642, 1655, 3109]

# 2. Manuálny mapping pre tieto konkrétne letiská (z tvojho predošlého zadania)
route_mapping_2 = {
    2909: (79, 75),  # Sinop (Turkey) - Hore
    863: (-10, -0),  # Panama - Vľavo
    860: (20, 70),    # Bocas Del Toro - Vpravo
    868: (180, 0),   # San Jose (Costa Rica) - Hore
    1744: (79, 75),  # Houston (USA) - Hore
    1642: (79, 70),   # Beijing (China) - Vpravo
    1655: (185, 32), # Xi'an (China) - Vľavo
    3109: (79, 85),  # Mackenzie (Canada) - Hore
}

# 3. Vytvorenie DataFrame pre uzly (Nodes)
route_nodes_2 = nodes.loc[node_indices].copy()
# 1. Priradenie kotvy cez index (idx je index v DataFrame, ktorý zodpovedá route_mapping)
route_nodes_2["anchor"] = route_nodes_2.index.map(route_mapping_2)

# 2. Ak by nejaký index v slovníku chýbal, dáme mu stredovú kotvu
route_nodes_2["anchor"] = route_nodes_2["anchor"].fillna(
    {i: (79, 40) for i in route_nodes_2.index}
)

# 3. Vykreslenie
rm = RouteMap()
rm.add_nodes(route_nodes_2, show_text=True)


route_edges = edges.loc[zip(node_indices, node_indices[1:])]
route_edges = make_start_end(route_edges)
route_edges["src_city"] = route_edges["# source"].map(nodes["city"])
route_edges["dest_city"] = route_edges["target"].map(nodes["city"])

rm.add_route(route_edges, False)
rm.showMap("./vizs/distance_map.html")

In [8]:
route_edges

,# source,target,distance,start,end,src_city,dest_city
0,2909,863,11302.730990,"42.015800476074,35.066398620605","8.973340034484863,-79.55560302734375",Sinop,Panama
1,863,860,298.774291,"8.973340034484863,-79.55560302734375","9.340849876403809,-82.25080108642578",Panama,Bocas Del Toro
2,860,868,226.647594,"9.340849876403809,-82.25080108642578","9.993860244750977,-84.20880126953125",Bocas Del Toro,San Jose
3,868,1744,2505.695965,"9.993860244750977,-84.20880126953125","29.984399795532227,-95.34140014648438",San Jose,Houston
4,1744,1642,11559.932616,"29.984399795532227,-95.34140014648438","40.0801010131836,116.58499908447266",Houston,Beijing
5,1642,1655,933.833791,"40.0801010131836,116.58499908447266","34.447102,108.751999",Beijing,Xi'an
6,1655,3109,8887.536731,"34.447102,108.751999","55.304402,-123.132004",Xi'an,Mackenzie British Columbia
